In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 2
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
13.211612120366748

Trial 1 =========================================
16.171405276024192

Trial 2 =========================================
13.047793894398264

Trial 3 =========================================
16.003415382632575

Trial 4 =========================================
14.087010089112685



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 5 =========================================
15.960709674372405

Trial 6 =========================================
13.8313653848276

Trial 7 =========================================
13.445600429679727

Trial 8 =========================================
14.288406037714791

Trial 9 =========================================
13.44186755160799

Trial 10 =========================================
15.325991232175372

Trial 11 =========================================
15.894390760417197

Trial 12 =========================================
18.641319894866424

Trial 13 =========================================
15.396502436136137

Trial 14 =========================================
15.733148312530144

Trial 15 =========================================
15.375378352015085

Trial 16 =========================================
15.384563255751111

Trial 17 =========================================
13.457247184034625

Trial 18 =========================================
18.640215983160495

Trial 19 =====

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 56 =========================================
15.929222010399386



...

Trial 60 =========================================
16.095873475170833

Trial 61 =========================================
13.447525043978821

Trial 62 =========================================
15.383492264430947

Trial 63 =========================================
14.405689531208957

Trial 64 =========================================
13.635589799085475

Trial 65 =========================================
14.94049426480239

Trial 66 =========================================
17.59164440641576

Trial 67 =========================================
15.379244389749104

Trial 68 =========================================
17.55152830243906

Trial 69 =========================================
16.195952731329246

Trial 70 =========================================
17.437188272494012

Trial 71 =========================================
12.439892855105716

Trial 72 =========================================
18.769782307409695

Trial 73 =========================================
14.630734653252189



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 74 =========================================
15.977382063719459

Trial 75 =========================================
13.171901060100447

Trial 76 =========================================
14.854169750009108

Trial 77 =========================================
13.45484380372405

Trial 78 =========================================
15.929222010399386

Trial 79 =========================================
14.270604311610152

Trial 80 =========================================
14.557951318531085

Trial 81 =========================================
12.740534238557137



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 82 =========================================
16.09073093203238

Trial 83 =========================================
15.891693569579337

Trial 84 =========================================
18.81991514058585

Trial 85 =========================================
16.237825216345588

Trial 86 =========================================
15.381724442594656

Trial 87 =========================================
15.395503956976004

Trial 88 =========================================
15.372296036939694

Trial 89 =========================================
18.795603131080398

Trial 90 =========================================
13.445915203829955

Trial 91 =========================================
15.396830571158334

Trial 92 =========================================
15.359382230608755

Trial 93 =========================================
13.43889322626767

Trial 94 =========================================
15.390693777960953

Trial 95 =========================================
15.37942326687395

Trial 96 =

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.81991514058585
Avg = 15.26117527947694
Std = 1.495008747610167


In [6]:
print(y_max_arr.tolist())

[13.211612120366748, 16.171405276024192, 13.047793894398264, 16.003415382632575, 14.087010089112685, 15.960709674372405, 13.8313653848276, 13.445600429679727, 14.288406037714791, 13.44186755160799, 15.325991232175372, 15.894390760417197, 18.641319894866424, 15.396502436136137, 15.733148312530144, 15.375378352015085, 15.384563255751111, 13.457247184034625, 18.640215983160495, 14.627770979785145, 14.269064566012235, 15.390125759729482, 16.11756354432642, 15.390020969032637, 13.879503261651605, 14.204501346030728, 14.281781420164686, 14.439058753596377, 16.243271031521328, 15.399886476964948, 16.191424258153784, 15.397607139744302, 15.386804024135866, 13.496930109472562, 15.940692690642152, 14.285188420020896, 14.748100768730902, 15.383519202548134, 13.176992901428555, 14.62644743360286, 15.726786884467634, 15.889096004425525, 18.762762363712575, 13.834964463231799, 16.250581618399845, 18.724771347909243, 16.237917429421522, 16.077981136844134, 13.017914871823141, 13.51027961863077, 16.17

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-2.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)